# 🔴 Solution: Top-p Sampling

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def top_p_sample(logits: torch.Tensor, p: float) -> torch.Tensor:
    squeeze = logits.dim() == 1
    if squeeze:
        logits = logits.unsqueeze(0)

    probs = torch.softmax(logits, dim=-1)
    outputs = []
    for row in probs:
        sorted_probs, sorted_indices = torch.sort(row, descending=True)
        cumulative = sorted_probs.cumsum(dim=0)
        keep = cumulative <= p
        keep[0] = True

        first_exceed = torch.nonzero(cumulative > p, as_tuple=False)
        if first_exceed.numel() > 0:
            keep[first_exceed[0, 0]] = True

        filtered = torch.zeros_like(row)
        filtered[sorted_indices[keep]] = row[sorted_indices[keep]]
        filtered = filtered / filtered.sum()
        outputs.append(torch.multinomial(filtered, num_samples=1).squeeze(0))

    outputs = torch.stack(outputs)
    return outputs.squeeze(0) if squeeze else outputs


In [ ]:
# Verify
torch.manual_seed(0)
logits = torch.tensor([[4.0, 3.5, 1.0, -2.0]])
print(top_p_sample(logits, p=0.8))
